In [1]:
import torch     
import torch.nn as nn      

class MultiQueryAttention(nn.Module):
  def __init__(self, d_in, d_out, num_heads, dropout=0.0):
    super().__init__()
    assert d_out % num_heads == 0, "d_model must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads

    self.W_query = nn.Linear(d_in, d_out, bias=False)
    self.W_key = nn.Linear(d_in, self.head_dim , bias=False)    # Single projection for k
    self.W_value = nn.Linear(d_in, self.head_dim , bias=False)  # Single projection for V

    self.out_proj = nn.Linear(d_out, d_out)

    self.dropout = nn.Dropout(dropout)

    # Using a fixed size mask for demonstration. A dynamic one is better in practice.
    self.register_buffer("mask", torch.triu(torch.ones(1, 1, 1024, 1024), diagonal=1))

  def forward(self, x):
    batch_size, num_tokens, d_in =  x.shape

    # Query
    queries = self.W_query(x)    # (batch_size, num_tokens, d_out)
    
    # Unroll last dim : (batch_size, num_tokens, d_out)  ---> (batch_size, num_tokens, num_heads, head_dim)
    queries = queries.view(batch_size, num_tokens, self.num_heads, self.head_dim)
    # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
    queries = queries.transpose(1, 2)

    print("queries")
    print(queries.shape)
    print(queries)

    
    # Key & value
    keys = self.W_key(x)      # (batch_size, num_tokens, head_dim)
    values = self.W_value(x)  # (batch_size, num_tokens, head_dim)

    # Unroll last dim : (batch_size, num_tokens, head_dim)  ---> (batch_size, num_tokens, 1, head_dim)
    keys = keys.view(batch_size, num_tokens, 1, self.head_dim)     # only 1 head
    values = values.view(batch_size, num_tokens, 1, self.head_dim)  # only 1 head

    # Transpose: (batch_size, num_tokens, 1, head_dim) -> (batch_size, 1, num_tokens, head_dim)
    keys = keys.transpose(1, 2)
    values = values.transpose(1, 2)

    print("keys")
    print(keys.shape)
    print(keys)

    # Now Repeat K and V to match the query head
    keys = keys.repeat(1, self.num_heads, 1, 1)  # (batch_size, num_heads, num_tokens, head_dim)
    values = values.repeat(1, self.num_heads, 1, 1)  # (batch_size, num_heads, num_tokens, head_dim)

    print("Repeat num heads of keys to match num heahds of query") # print same for values
    print(keys.shape)
    print(keys)
    

    # Attn scores
    attn_scores = queries @ keys.transpose(2, 3)

    # Apply causal mask
    # Original mask truncated to the number of tokens and converted to boolean
    mask_bool = self.mask.bool()[:,:,:num_tokens,:num_tokens]
    # Use the mask to fill attention scores
    attn_scores = attn_scores.masked_fill_(mask_bool, -torch.inf)

    # Attn weights
    attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)

    # dropout
    attn_weights = self.dropout(attn_weights)

    # context vectors
    context_vector = attn_weights @ values  # (batch_size, num_heads, num_tokens, head_dim)

    # (batch_size, num_tokens, num_heads, head_dim)
    context_vector = context_vector.transpose(1, 2)

    # Combine heads, where self.d_out = self.num_heads * self.head_dim
    # (batch_size, num_tokens, num_heads, head_dim) ---> # (batch_size, num_tokens, d_out)
    context_vector = context_vector.contiguous().view(batch_size, num_tokens, self.d_out)

    context_vector = self.out_proj(context_vector)

    return context_vector


c:\Users\Manoj\Python ML\Lib\site-packages\torch\cuda\__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
batch_size = 1
num_tokens = 6
d_in = 8

d_out = 8
# d_model = 4

mqa = MultiQueryAttention(d_in, d_out, num_heads=4, dropout=0.0)

inputs = torch.rand(batch_size, num_tokens, d_in)

context_vecs = mqa(inputs)

print("MQA Layer successful!")
print(f"Input shape: {inputs.shape}")
print(f"Output shape: {context_vecs.shape}")

# print(inputs)
print(context_vecs)

queries
torch.Size([1, 4, 6, 2])
tensor([[[[-0.2054,  0.5460],
          [-0.2458,  0.1744],
          [-0.4260,  0.3939],
          [-0.4876,  0.1423],
          [-0.6264,  0.3513],
          [-0.4880,  0.3431]],

         [[-0.3391,  0.3074],
          [-0.1703,  0.2390],
          [-0.2253,  0.3649],
          [-0.6016,  0.2969],
          [-0.2422,  0.4272],
          [-0.2167,  0.0467]],

         [[-0.2734, -0.1627],
          [-0.3304, -0.0424],
          [-0.3922, -0.1557],
          [ 0.1505,  0.0059],
          [ 0.0031, -0.2617],
          [-0.2518, -0.3248]],

         [[-0.5395,  0.1547],
          [-0.5126,  0.1483],
          [-0.5068,  0.5828],
          [ 0.1994,  0.1418],
          [-0.3299,  0.2375],
          [-0.3366,  0.4840]]]], grad_fn=<TransposeBackward0>)
keys
torch.Size([1, 1, 6, 2])
tensor([[[[-0.3981,  0.2610],
          [-0.1984,  0.1575],
          [-0.0803,  0.0486],
          [-0.0804, -0.1520],
          [-0.1073,  0.1875],
          [-0.2113, -0.1346]

As we can see from the above output that 

Query : parameters/values are different for each head

Keys and Values :  parameter/values are same for each head, because we are sharing for each head

Multi-Query Attention (MQA):

Standard multi-head attention requires unique K, V projections for each attention head, which increases the KV cache size proportionally to the number of heads. Multi-Query Attention addresses this by:

Using a single shared K, V projection across all attention heads

Maintaining separate Q projections for each head